## Porównanie z baseline i innymi modelami

### 📋 Cel notebooka
Ten notebook porównuje model BERT z:
- **Rule-based baseline** - proste reguły oparte na wartościach badań laboratoryjnych
- **Inne modele ML** - Random Forest, Logistic Regression, XGBoost
- **Table 3** - porównanie metryk (wymagane dla publikacji)

### 🔄 Wymagania
- `bert_triage_model_*` - wytrenowany model BERT (z notebooka 05)
- `corpus_val_*.txt` - **TYLKO zbiór walidacyjny** z notebooka 05
- `corpus_train_*.txt` - zbiór treningowy (dla treningu baseline models)

### ⚠️ WAŻNE: Data Leakage Prevention
- Używamy **TYLKO** zbioru walidacyjnego z notebooka 05 do ewaluacji
- Baseline models trenujemy na zbiorze treningowym z notebooka 05

### 📊 Metodologia
- **Rule-based baseline:** Proste reguły oparte na tokenach (CRITICAL, HIGH, LOW)
- **Random Forest:** Ensemble method z 100 drzewami
- **Logistic Regression:** Liniowy model z regularyzacją L2
- **XGBoost:** Gradient boosting z optymalizacją hiperparametrów
- **Metryki:** Accuracy, Precision, Recall, F1 (macro i weighted)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from collections import Counter
from transformers import BertForSequenceClassification, PreTrainedTokenizerFast
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from pathlib import Path
import glob
import sys
import warnings
warnings.filterwarnings('ignore')

# Konfiguracja wizualizacji
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

# ⚠️ AUTOMATYCZNE WYKRYWANIE NAJNOWSZYCH PLIKÓW
triage_models = sorted([d for d in glob.glob("bert_triage_model_*") if Path(d).is_dir()], reverse=True)
if not triage_models:
    raise FileNotFoundError("Nie znaleziono modelu bert_triage_model_*. Uruchom najpierw notebook 05_Fine_Tune_Model.ipynb")
MODEL_PATH = triage_models[0]
print(f"📂 Znaleziono model BERT: {MODEL_PATH}")

# Wczytanie train/val split
val_split_files = sorted(glob.glob("corpus_val_*.txt"), reverse=True)
train_split_files = sorted(glob.glob("corpus_train_*.txt"), reverse=True)
if not val_split_files:
    raise FileNotFoundError("Nie znaleziono corpus_val_*.txt. Uruchom najpierw notebook 05.")
if not train_split_files:
    raise FileNotFoundError("Nie znaleziono corpus_train_*.txt. Uruchom najpierw notebook 05.")

VAL_FILE = val_split_files[0]
TRAIN_FILE = train_split_files[0]
print(f"📂 Train: {TRAIN_FILE}")
print(f"📂 Val: {VAL_FILE}")

# Obsługa MPS (Apple Silicon) dla przyspieszenia
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"🚀 Device selected: {DEVICE}")
BATCH_SIZE = 64

# Mapowanie klas (wspólny moduł — szukamy project root, gdzie jest label_mapping.py)
import os
current_dir = Path(os.getcwd())
project_root = None
if (current_dir / "label_mapping.py").exists():
    project_root = current_dir
elif (current_dir / "notebooks_ethos" / "label_mapping.py").exists():
    project_root = current_dir.parent
else:
    for parent in current_dir.parents:
        if (parent / "label_mapping.py").exists():
            project_root = parent
            break
if project_root is None:
    raise FileNotFoundError(
        f"Nie znaleziono pliku label_mapping.py. "
        f"Szukano w: {current_dir} i katalogach nadrzędnych."
    )
sys.path.insert(0, str(project_root))
print(f"📂 Dodano do sys.path: {project_root}")
from label_mapping import (
    LABEL_TO_ID,
    ID_TO_LABEL_SHORT,
    NUM_LABELS
)

print(f"🚀 Device: {DEVICE}")
print(f"📊 Liczba klas: {NUM_LABELS}")

In [ ]:
# Wczytanie danych treningowych i walidacyjnych
def load_data(filepath):
    texts = []
    labels = []
    raw_sequences = []
    
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            parts = line.split()
            if len(parts) < 2:
                continue
            
            label_str = parts[-1]
            input_text = " ".join(parts[:-1])
            
            if label_str in LABEL_TO_ID:
                texts.append(input_text)
                labels.append(LABEL_TO_ID[label_str])
                raw_sequences.append(line)
    
    return texts, labels, raw_sequences

print("📂 Wczytywanie danych treningowych...")
train_texts, train_labels, train_sequences = load_data(TRAIN_FILE)
print(f"✅ Train: {len(train_texts)} próbek")

print("📂 Wczytywanie danych walidacyjnych...")
val_texts, val_labels, val_sequences = load_data(VAL_FILE)
print(f"✅ Val: {len(val_texts)} próbek")

print(f"\n📊 Rozkład klas w train: {Counter([ID_TO_LABEL_SHORT[l] for l in train_labels])}")
print(f"📊 Rozkład klas w val: {Counter([ID_TO_LABEL_SHORT[l] for l in val_labels])}")

## 📋 1. RULE-BASED BASELINE (Poprawiony)

Proste reguły oparte na tokenach (dostosowane do kwantyli Q1-Q10):
1. **NEFROLOG**: Jeśli `CREATININE/UREA` są w `Q9` lub `Q10` (bardzo wysokie).
2. **HEMATOLOG**: Jeśli `HGB/PLT` są w `Q1` lub `Q2` (bardzo niskie/anemia/małopłytkowość).
3. **GASTROLOG**: Jeśli występuje `SYMPTOM_GI_BLEED_YES`.
4. **SOR**: Pozostałe przypadki z tokenem `_CRITICAL` (nie łapane przez powyższe).
5. **POZ**: Wszystko inne.

In [ ]:
def rule_based_classifier(sequence):
    """Rule-based classifier oparty na tokenach (poprawiona logika kwantyli i priorytetów)."""
    tokens = sequence.split()
    # Konwertujemy na string dla łatwiejszego wyszukiwania podciągów
    token_str = " ".join(tokens)
    
    # Priorytet 1: NEFROLOG (Silny sygnał: wysoka kreatynina/mocznik - Q9, Q10)
    # Złapie CREATININE_Q9, CREATININE_Q10_CRITICAL itp.
    if 'CREATININE_Q9' in token_str or 'CREATININE_Q10' in token_str or \
       'UREA_Q9' in token_str or 'UREA_Q10' in token_str:
        return LABEL_TO_ID['TARGET_SPEC_NEFROLOG']

    # Priorytet 2: HEMATOLOG (Silny sygnał: niska morfologia - Q1, Q2)
    # HGB_Q1, PLT_Q1 często oznaczają białaczkę/anemię
    if 'HGB_Q1' in token_str or 'HGB_Q2' in token_str or \
       'PLT_Q1' in token_str or 'PLT_Q2' in token_str:
        return LABEL_TO_ID['TARGET_SPEC_HEMATOLOG']
    
    # Priorytet 3: GASTROLOG (krwawienie GI)
    if 'SYMPTOM_GI_BLEED_YES' in token_str:
        return LABEL_TO_ID['TARGET_SPEC_GASTROLOG']
        
    # Priorytet 4: SOR (Inne stany krytyczne, które nie wpadły w powyższe)
    # Jeśli coś jest CRITICAL, a nie jest nerkowe/hematologiczne, to SOR
    if '_CRITICAL' in token_str:
        return LABEL_TO_ID['TARGET_SPEC_SOR']
    
    # Priorytet 5: Domyślnie POZ
    return LABEL_TO_ID['TARGET_POZ']

print("🔧 Rule-based baseline - predykcje na zbiorze walidacyjnym...")
rule_predictions = []
for seq in val_sequences:
    # Usuń etykietę z sekwencji (tylko tokeny)
    tokens_only = " ".join(seq.split()[:-1])
    pred = rule_based_classifier(tokens_only)
    rule_predictions.append(pred)

rule_predictions = np.array(rule_predictions)
val_labels_array = np.array(val_labels)

rule_accuracy = accuracy_score(val_labels_array, rule_predictions)
rule_precision, rule_recall, rule_f1, _ = precision_recall_fscore_support(
    val_labels_array, rule_predictions, average='weighted', zero_division=0
)

print("✅ Rule-based baseline:")
print(f"   Accuracy: {rule_accuracy:.3f}")
print(f"   Precision: {rule_precision:.3f}")
print(f"   Recall: {rule_recall:.3f}")
print(f"   F1: {rule_f1:.3f}")

## 🤖 2. INNE MODELE ML

Trenujemy proste modele ML na ekstrahowanych features z sekwencji:
- **Random Forest** - ensemble method
- **Logistic Regression** - liniowy model
- **XGBoost** - gradient boosting (jeśli dostępny)

In [ ]:
# Ekstrakcja features: tokeny jako binary features
print("🔍 Ekstrakcja features z sekwencji...")

# Zbierz wszystkie unikalne tokeny z train
all_tokens = set()
for seq in train_sequences:
    tokens = seq.split()[:-1]  # Bez etykiety
    all_tokens.update(tokens)

# Sortuj dla spójności
vocab = sorted(list(all_tokens))
vocab_dict = {token: idx for idx, token in enumerate(vocab)}

print(f"📊 Vocab size: {len(vocab)} tokenów")

def extract_features(sequences):
    """Ekstrahuje binary features z sekwencji."""
    features = []
    for seq in sequences:
        tokens = seq.split()[:-1]  # Bez etykiety
        feature_vector = np.zeros(len(vocab))
        for token in tokens:
            if token in vocab_dict:
                feature_vector[vocab_dict[token]] = 1
        features.append(feature_vector)
    return np.array(features)

print("🔧 Ekstrakcja features dla train...")
X_train = extract_features(train_sequences)
y_train = np.array(train_labels)

print("🔧 Ekstrakcja features dla val...")
X_val = extract_features(val_sequences)
y_val = np.array(val_labels)

print(f"✅ Train features: {X_train.shape}")
print(f"✅ Val features: {X_val.shape}")

In [ ]:
# Random Forest
print("🌲 Trening Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_val)
rf_accuracy = accuracy_score(y_val, rf_predictions)
rf_precision, rf_recall, rf_f1, _ = precision_recall_fscore_support(
    y_val, rf_predictions, average='weighted', zero_division=0
)

print("✅ Random Forest:")
print(f"   Accuracy: {rf_accuracy:.3f}")
print(f"   Precision: {rf_precision:.3f}")
print(f"   Recall: {rf_recall:.3f}")
print(f"   F1: {rf_f1:.3f}")

In [ ]:
# Logistic Regression (Safe Mode for Mac)
print("📊 Trening Logistic Regression...")
# Zmiana solvera na 'liblinear' - jest stabilniejszy na macOS i mniej pamięciożerny
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver='liblinear',  # Fix: Zmiana z lbfgs na liblinear
    multi_class='ovr'    # One-vs-Rest
)
lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_val)
lr_accuracy = accuracy_score(y_val, lr_predictions)
lr_precision, lr_recall, lr_f1, _ = precision_recall_fscore_support(
    y_val, lr_predictions, average='weighted', zero_division=0
)

print("✅ Logistic Regression:")
print(f"   Accuracy: {lr_accuracy:.3f}")
print(f"   Precision: {lr_precision:.3f}")
print(f"   Recall: {lr_recall:.3f}")
print(f"   F1: {lr_f1:.3f}")

In [ ]:
# XGBoost (Safe Mode)
try:
    import xgboost as xgb
    print("🚀 Trening XGBoost...")
    print("   ⏱️ 173k próbek – może zająć chwilę. Limit wątków CPU ustawiony na 4.")
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        random_state=42,
        objective='multi:softmax',
        num_class=NUM_LABELS,
        n_jobs=4  # Fix: Ograniczamy wątki, żeby nie zabić kernela
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=10,
    )
    
    xgb_predictions = xgb_model.predict(X_val)
    xgb_accuracy = accuracy_score(y_val, xgb_predictions)
    xgb_precision, xgb_recall, xgb_f1, _ = precision_recall_fscore_support(
        y_val, xgb_predictions, average='weighted', zero_division=0
    )
    
    print("✅ XGBoost:")
    print(f"   Accuracy: {xgb_accuracy:.3f}")
    print(f"   Precision: {xgb_precision:.3f}")
    print(f"   Recall: {xgb_recall:.3f}")
    print(f"   F1: {xgb_f1:.3f}")
    
    XGBOOST_AVAILABLE = True
except ImportError:
    print("⚠️ XGBoost nie jest dostępny. Pomijam.")
    XGBOOST_AVAILABLE = False
except Exception as e:
    print(f"⚠️ Błąd podczas treningu XGBoost: {e}")
    XGBOOST_AVAILABLE = False

## 🧠 3. MODEL BERT

Wykonujemy predykcje używając wytrenowanego modelu BERT.

In [ ]:
# Wczytanie modelu BERT i wykonanie predykcji
print("🧠 Ładowanie modelu BERT...")
tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_PATH, local_files_only=True)
bert_model = BertForSequenceClassification.from_pretrained(MODEL_PATH, local_files_only=True)
bert_model.to(DEVICE)
bert_model.eval()

class EthosEvalDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten()}

print("🔮 Wykonywanie predykcji BERT...")
bert_dataset = EthosEvalDataset(val_texts, tokenizer)
bert_dataloader = DataLoader(bert_dataset, batch_size=BATCH_SIZE, shuffle=False)

bert_predictions = []
with torch.no_grad():
    for batch in tqdm(bert_dataloader, desc="BERT predykcje"):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)
        bert_predictions.extend(predictions.cpu().numpy())

bert_predictions = np.array(bert_predictions)
bert_accuracy = accuracy_score(y_val, bert_predictions)
bert_precision, bert_recall, bert_f1, _ = precision_recall_fscore_support(
    y_val, bert_predictions, average='weighted', zero_division=0
)

print("✅ BERT:")
print(f"   Accuracy: {bert_accuracy:.3f}")
print(f"   Precision: {bert_precision:.3f}")
print(f"   Recall: {bert_recall:.3f}")
print(f"   F1: {bert_f1:.3f}")

## 📊 TABLE 3: PORÓWNANIE Z BASELINE

Tabela porównująca wszystkie modele (wymagana dla publikacji).

In [ ]:
# Tworzenie Table 3
results = {
    'Model': ['Rule-based', 'Random Forest', 'Logistic Regression'],
    'Accuracy': [rule_accuracy, rf_accuracy, lr_accuracy],
    'Precision': [rule_precision, rf_precision, lr_precision],
    'Recall': [rule_recall, rf_recall, lr_recall],
    'F1': [rule_f1, rf_f1, lr_f1]
}

if XGBOOST_AVAILABLE:
    results['Model'].append('XGBoost')
    results['Accuracy'].append(xgb_accuracy)
    results['Precision'].append(xgb_precision)
    results['Recall'].append(xgb_recall)
    results['F1'].append(xgb_f1)

results['Model'].append('BERT (proposed)')
results['Accuracy'].append(bert_accuracy)
results['Precision'].append(bert_precision)
results['Recall'].append(bert_recall)
results['F1'].append(bert_f1)

df_table3 = pd.DataFrame(results)
df_table3 = df_table3.round(3)

print("📊 TABLE 3: PORÓWNANIE Z BASELINE\n")
print(df_table3.to_string(index=False))

# Zapis do CSV
df_table3.to_csv('table3_baseline_comparison.csv', index=False)
print("\n💾 Zapisano do: table3_baseline_comparison.csv")

## 🔬 Test McNemara: porównanie modeli (BERT vs baseline)

Test McNemara służy do porównania **dwóch klasyfikatorów na tym samym zbiorze** (paired nominal data). Sprawdzamy, czy różnica w liczbie próbek, gdzie jeden model ma rację a drugi się myli, jest istotna statystycznie.

- **Tabela 2×2:** (BERT poprawny, baseline poprawny) vs (BERT poprawny, baseline błędny) vs (BERT błędny, baseline poprawny) vs (oba błędne).
- **H0:** Proporcja par dyskordantnych (BERT✓ RF✗ vs BERT✗ RF✓) jest 50:50 — modele nie różnią się istotnie.
- **Wynik:** chi², p-value; jeśli p < 0,05, uznajemy że modele różnią się istotnie.

In [ ]:
from scipy.stats import mcnemar

def mcnemar_two_models(y_true, y_pred_a, y_pred_b, name_a="Model A", name_b="Model B"):
    """
    Test McNemara dla dwóch modeli na tym samym zbiorze.
    Tabela: [obaj błędni, A błędny B poprawny; A poprawny B błędny, obaj poprawni].
    """
    correct_a = (np.array(y_pred_a) == np.array(y_true))
    correct_b = (np.array(y_pred_b) == np.array(y_true))
    n_00 = np.sum(~correct_a & ~correct_b)  # obaj błędni
    n_01 = np.sum(~correct_a & correct_b)    # A błędny, B poprawny
    n_10 = np.sum(correct_a & ~correct_b)  # A poprawny, B błędny
    n_11 = np.sum(correct_a & correct_b)    # obaj poprawni
    table = np.array([[n_00, n_01], [n_10, n_11]])
    # scipy mcnemar: table[0,1]=A wrong B right, table[1,0]=A right B wrong
    try:
        result = mcnemar(table, exact=False, correction=True)
        chi2 = result.statistic if hasattr(result, "statistic") else result[0]
        pval = result.pvalue if hasattr(result, "pvalue") else result[1]
    except Exception:
        chi2, pval = np.nan, np.nan
    return {
        "model_a": name_a, "model_b": name_b,
        "n_both_wrong": n_00, "n_a_wrong_b_right": n_01, "n_a_right_b_wrong": n_10, "n_both_right": n_11,
        "chi2": chi2, "p_value": pval,
        "significant": pval < 0.05 if not np.isnan(pval) else False
    }

# Porównanie BERT vs każdy baseline
mcnemar_results_models = []

# BERT vs Random Forest
r = mcnemar_two_models(y_val, bert_predictions, rf_predictions, "BERT", "Random Forest")
mcnemar_results_models.append(r)
print("🔬 McNemar: BERT vs Random Forest")
print(f"   Tabela 2×2: obaj błędni={r['n_both_wrong']}, BERT✗ RF✓={r['n_a_wrong_b_right']}, BERT✓ RF✗={r['n_a_right_b_wrong']}, obaj poprawni={r['n_both_right']}")
print(f"   chi²={r['chi2']:.4f}, p-value={r['p_value']:.6f} → {'Istotna różnica (p<0.05)' if r['significant'] else 'Brak istotnej różnicy'}")

# BERT vs Logistic Regression
r = mcnemar_two_models(y_val, bert_predictions, lr_predictions, "BERT", "Logistic Regression")
mcnemar_results_models.append(r)
print("\n🔬 McNemar: BERT vs Logistic Regression")
print(f"   Tabela 2×2: obaj błędni={r['n_both_wrong']}, BERT✗ LR✓={r['n_a_wrong_b_right']}, BERT✓ LR✗={r['n_a_right_b_wrong']}, obaj poprawni={r['n_both_right']}")
print(f"   chi²={r['chi2']:.4f}, p-value={r['p_value']:.6f} → {'Istotna różnica (p<0.05)' if r['significant'] else 'Brak istotnej różnicy'}")

if XGBOOST_AVAILABLE:
    r = mcnemar_two_models(y_val, bert_predictions, xgb_predictions, "BERT", "XGBoost")
    mcnemar_results_models.append(r)
    print("\n🔬 McNemar: BERT vs XGBoost")
    print(f"   Tabela 2×2: obaj błędni={r['n_both_wrong']}, BERT✗ XGB✓={r['n_a_wrong_b_right']}, BERT✓ XGB✗={r['n_a_right_b_wrong']}, obaj poprawni={r['n_both_right']}")
    print(f"   chi²={r['chi2']:.4f}, p-value={r['p_value']:.6f} → {'Istotna różnica (p<0.05)' if r['significant'] else 'Brak istotnej różnicy'}")

# Zapis do CSV
df_mcnemar_models = pd.DataFrame([
    {
        "Model A": x["model_a"], "Model B": x["model_b"],
        "chi2": x["chi2"], "p_value": x["p_value"],
        "Istotne (p<0.05)": "Tak" if x["significant"] else "Nie",
        "BERT✗ B✓": x["n_a_wrong_b_right"], "BERT✓ B✗": x["n_a_right_b_wrong"]
    }
    for x in mcnemar_results_models
])
df_mcnemar_models.to_csv("mcnemar_models_comparison.csv", index=False, encoding="utf-8-sig")
print("\n✅ Wyniki zapisane do: mcnemar_models_comparison.csv")

In [ ]:
# Wizualizacja porównania
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    ax.bar(df_table3['Model'], df_table3[metric], color=['steelblue', 'coral', 'lightgreen', 'orange', 'purple'][:len(df_table3)])
    ax.set_title(f'{metric} Comparison', fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylim([0, 1])
    
    # Zaznacz najlepszy wynik
    best_idx = df_table3[metric].idxmax()
    ax.bar(best_idx, df_table3.loc[best_idx, metric], color='gold', alpha=0.7)

plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("💾 Zapisano wykres: baseline_comparison.png")

In [ ]:
# Podsumowanie
print("📋 PODSUMOWANIE PORÓWNANIA\n")
print("=" * 60)
print(f"Najlepszy Accuracy: {df_table3.loc[df_table3['Accuracy'].idxmax(), 'Model']} ({df_table3['Accuracy'].max():.3f})")
print(f"Najlepszy F1: {df_table3.loc[df_table3['F1'].idxmax(), 'Model']} ({df_table3['F1'].max():.3f})")
print("\nBERT vs Rule-based:")
print(f"   Accuracy: {bert_accuracy - rule_accuracy:+.3f}")
print(f"   F1: {bert_f1 - rule_f1:+.3f}")
if XGBOOST_AVAILABLE:
    print("\nBERT vs XGBoost:")
    print(f"   Accuracy: {bert_accuracy - xgb_accuracy:+.3f}")
    print(f"   F1: {bert_f1 - xgb_f1:+.3f}")
print("=" * 60)